1. Kütüphane Yükleme
Bu blokta, veri manipülasyonu, makine öğrenimi modelleme ve performans değerlendirmesi için gerekli temel Python kütüphaneleri yükleniyor.

In [1]:

# Temel kütüphanelerin ve model bileşenlerinin yüklenmesi

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

2. Veri Yükleme ve İlk İnceleme
İki farklı CSV dosyası (googleplaystore.csv ve googleplaystore_user_reviews.csv) DataFrame'lere yükleniyor ve temel bilgiler (boyut ve ilk 5 satır) ekrana yazdırılıyor.

In [2]:
# Veri setlerinin yüklenmesi
df = pd.read_csv(r"dataset/googleplaystore.csv")
reviews_df = pd.read_csv(r"dataset/googleplaystore_user_reviews.csv")

# Temel veri özetleri
print("Apps Shape:", df.shape)
print("Reviews Shape:", reviews_df.shape)

# İlk gözlemler
print(df.head())
print(reviews_df.head())

Apps Shape: (10841, 13)
Reviews Shape: (64295, 5)
                                                 App        Category  Rating  \
0     Photo Editor & Candy Camera & Grid & ScrapBook  ART_AND_DESIGN     4.1   
1                                Coloring book moana  ART_AND_DESIGN     3.9   
2  U Launcher Lite – FREE Live Cool Themes, Hide ...  ART_AND_DESIGN     4.7   
3                              Sketch - Draw & Paint  ART_AND_DESIGN     4.5   
4              Pixel Draw - Number Art Coloring Book  ART_AND_DESIGN     4.3   

  Reviews  Size     Installs  Type Price Content Rating  \
0     159   19M      10,000+  Free     0       Everyone   
1     967   14M     500,000+  Free     0       Everyone   
2   87510  8.7M   5,000,000+  Free     0       Everyone   
3  215644   25M  50,000,000+  Free     0           Teen   
4     967  2.8M     100,000+  Free     0       Everyone   

                      Genres      Last Updated         Current Ver  \
0               Art & Design   January 7, 20

3. Veri Birleştirme, Temizleme ve Hedef Değişken Oluşturma: Bu blokta ana veri setleri birleştiriliyor, temizleniyor ve sınıflandırma için bir Hedef Değişken (Success) oluşturuluyor.Duygu Analizi Verisi Birleştirme: Kullanıcı yorumlarındaki ortalama Sentiment_Polarity ve Sentiment_Subjectivity değerleri hesaplanarak ana uygulama verisine (df) sol birleştirme (left merge) ile ekleniyor.Tekrar Eden Kayıtları Temizleme: Uygulamalar Last Updated (Son Güncelleme) tarihine göre sıralanıp, tekrar eden uygulama isimlerinden sadece en güncel olanı tutuluyor (drop_duplicates).Eksik Derecelendirme (Rating) Silme: Modelin hedef değişkeni Rating'e dayandığı için, Rating sütunundaki eksik değer içeren satırlar siliniyor.Hedef Değişken (Success) Oluşturma: Uygulamanın başarılı olup olmadığını belirlemek için ikili (binary) bir sınıflandırma değişkeni oluşturuluyor:Rating 4.0 ve üzeriyse = 1 (Başarılı) Aksi takdirde = 0 (Başarısız)

In [3]:
# Duygu ortalamaları
sent_agg = reviews_df.groupby("App")[["Sentiment_Polarity", "Sentiment_Subjectivity"]].mean().reset_index()
sent_agg.columns = ["App", "Avg_Sentiment_Polarity", "Avg_Sentiment_Subjectivity"]

# Veri birleştirme
df = df.merge(sent_agg, on="App", how="left")

# Tarih işlemesi
df["Last Updated"] = pd.to_datetime(df["Last Updated"], errors="coerce")
df = df.sort_values("Last Updated", ascending=False).drop_duplicates(subset=["App"], keep="first")

# Rating temizleme
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
df = df.dropna(subset=["Rating"])

# Hedef değişkeni
df["Success"] = (df["Rating"] >= 4.0).astype(int)

# Dağılım özeti
print("\nTarget distribution (Success):")
print(df["Success"].value_counts(normalize=True))


Target distribution (Success):
Success
1    0.76711
0    0.23289
Name: proportion, dtype: float64


Çıktı Yorumu: Hedef değişken dengesiz (imbalanced). Başarılı (1) sınıfı, verinin yaklaşık %77'sini oluşturuyor.

4. Özellik Mühendisliği (Feature Engineering):
- Bu fonksiyon, modelin kullanacağı özellikleri oluşturmak ve mevcut özellikleri temizleyip sayısal formata dönüştürmek için tanımlanmıştır.Temizleme İşlemleri: Reviews, Installs ve Price sütunları sayısal değere dönüştürülürken, metinsel ifadeler ('+', ',', '$') kaldırılıyor.Size Dönüşümü: Megabayt ('M') ve Kilobayt ('k') ifadeleri temizleniyor ve 'Varies with device' (Cihaza göre değişir) değerleri eksik değer (np.nan) olarak işaretleniyor.Yeni Özellikler:Review_Rate: Uygulama başına düşen yorum oranı Reviews / (Installs + 1).Recency_Days: Son güncellemeden günümüze (1 Ağustos 2018) kadar geçen süre (gün cinsinden).Price_Category: Price sütununu kategorik gruplara ayıran yeni bir özellik (ücretsiz, ucuz, düşük, orta, yüksek).

In [4]:
def clean_and_engineer_features(df_):
    df_ = df_.copy()

    # Yorum sayısı
    df_["Reviews"] = pd.to_numeric(df_["Reviews"], errors="coerce")

    # Kurulum sayısı
    df_["Installs"] = (
        df_["Installs"]
        .astype(str)
        .str.replace("+", "", regex=False)
        .str.replace(",", "", regex=False)
    )
    df_["Installs"] = pd.to_numeric(df_["Installs"], errors="coerce")

    # Fiyat düzenleme
    df_["Price"] = df_["Price"].astype(str).str.replace("$", "", regex=False)
    df_["Price"] = pd.to_numeric(df_["Price"], errors="coerce")

    # Boyut temizleme
    size = df_["Size"].astype(str)
    size = size.str.replace("M", "", regex=False)
    size = size.str.replace("k", "", regex=False)
    size = size.replace("Varies with device", np.nan)
    df_["Size"] = pd.to_numeric(size, errors="coerce")

    # İnceleme oranı
    df_["Review_Rate"] = df_["Reviews"] / (df_["Installs"] + 1)

    # Güncellik hesaplama
    TODAY = pd.to_datetime("2018-08-01")
    df_["Last Updated"] = pd.to_datetime(df_["Last Updated"], errors="coerce")
    df_["Recency_Days"] = (TODAY - df_["Last Updated"]).dt.days

    # Fiyat kategorisi
    df_["Price_Category"] = pd.cut(
        df_["Price"].fillna(0),
        bins=[-0.01, 0, 1, 5, 20, np.inf],
        labels=["free", "cheap", "low", "medium", "high"]
    )

    return df_

df = clean_and_engineer_features(df)

5. Veri Setini Hazırlama ve Bölme:
Hedef değişken (Success) hariç tutularak özellik matrisi X ve hedef vektörü y belirleniyor. Ardından, veriler eğitim (X_train, y_train) ve test (X_test, y_test) kümelerine bölünüyor. Bu bölme işlemi, hedef değişkenin dengesizliği nedeniyle stratified (katmanlı) olarak yapılıyor.

In [5]:
# Değişken seçimi
drop_cols = ["App", "Rating", "Success", "Current Ver", "Android Ver", "Last Updated"]
drop_cols = [c for c in drop_cols if c in df.columns]

# Özellik-hedef ayrımı
X = df.drop(columns=drop_cols)
y = df["Success"]

# Veri bölünmesi
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Boyut bilgisi
print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)



Train shape: (6557, 13)
Test shape: (1640, 13)


6. Ön İşleme Hattı (Preprocessing Pipeline) Oluşturma:
Model eğitimi öncesinde gerekli veri temizleme (imputation), ölçekleme (scaling) ve kodlama (encoding) adımlarını otomatik olarak uygulayacak bir ColumnTransformer oluşturuluyor.

Sayısal Özellikler için İşlemler:

SimpleImputer(strategy="median"): Eksik değerleri medyan ile doldur.

StandardScaler(): Veriyi standartlaştır (ortalama 0, standart sapma 1).

Kategorik Özellikler için İşlemler:

SimpleImputer(strategy="most_frequent"): Eksik değerleri en sık görülen değer ile doldur.

OneHotEncoder(handle_unknown="ignore"): Kategorik değerleri One-Hot Encoding ile sayısal hale getir.

In [6]:
# Değişken türleri
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

# Tür listeleri
print("\nNumeric features:", numeric_features)
print("Categorical features:", categorical_features)

# Sayısal dönüşüm
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

# Kategorik dönüşüm
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

# Ön işleme yapısı
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)


Numeric features: ['Reviews', 'Size', 'Installs', 'Price', 'Avg_Sentiment_Polarity', 'Avg_Sentiment_Subjectivity', 'Review_Rate', 'Recency_Days']
Categorical features: ['Category', 'Type', 'Content Rating', 'Genres', 'Price_Category']


7. Özellik Seçimi (Feature Selection):
- HazırlığıSelectKBest metodu, istatistiksel testlere dayanarak en iyi K tane özelliği seçmek için kullanılacak. f_classif (ANOVA F-değeri), sınıflandırma problemlerinde bağımlı değişkenle en alakalı özellikleri bulmak için kullanılır. Burada K=75 olarak belirleniyor.

In [7]:
# Özellik seçimi
K = 75

# ANOVA filtresi
anova_selector = SelectKBest(score_func=f_classif, k=K)

8. Modeller ve Değerlendirme Fonksiyonu:
Bu blokta kullanılacak üç doğrusal model (LogisticRegression, SGDClassifier, LinearSVC) tanımlanıyor ve modelleri eğiten, tahmin yapan ve performans metriklerini hesaplayan bir fonksiyon oluşturuluyor.

Model Seçimi Notu: Sınıf dengesizliğini ele almak için LogisticRegression ve LinearSVC'ye varsayılan olarak class_weight="balanced" parametresi eklenmiş (SGDClassifier için sonraki adımlarda bu parametre ayarlanacaktır).

evaluate_model Fonksiyonu Notu: roc_auc_score'u hesaplamak için modelin olasılık (predict_proba) veya karar fonksiyonu (decision_function) çıktısını kullanıp kullanamadığı kontrol ediliyor. LinearSVC gibi bazı modeller olasılık yerine karar skorları verir. SGDClassifier'da loss="log_loss" kullanıldığında, aslında L2 düzenlemesi olan bir Lojistik Regresyon eğitilir ve bu da olasılık çıktısı verir.

In [8]:
# Modellerin tanımı
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=42
    ),
    "SGDClassifier": SGDClassifier(
        loss="log_loss", max_iter=2000, alpha=1e-4, random_state=42
    ),
    "LinearSVC": LinearSVC(
        random_state=42
    ),
}

# Temel boru hattı
def make_baseline_pipeline(clf):
    """Önişleme + sınıflandırıcı."""
    return Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("clf", clf),
        ]
    )

# ANOVA’lı boru hattı
def make_anova_pipeline(clf):
    """Önişleme + ANOVA + model."""
    return Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("select", anova_selector),
            ("clf", clf),
        ]
    )

# Model değerlendirme
def evaluate_model(pipeline, X_tr, X_te, y_tr, y_te, model_name, variant):
    pipeline.fit(X_tr, y_tr)
    y_pred = pipeline.predict(X_te)

    # Skor olasılıkları
    if hasattr(pipeline, "predict_proba"):
        y_score = pipeline.predict_proba(X_te)[:, 1]
    elif hasattr(pipeline, "decision_function"):
        y_score = pipeline.decision_function(X_te)
    else:
        y_score = None

    # Temel ölçütler
    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred)
    rec = recall_score(y_te, y_pred)
    f1 = f1_score(y_te, y_pred)
    if y_score is not None:
        roc = roc_auc_score(y_te, y_score)
    else:
        roc = np.nan

    # Sonuç çıktısı
    print(f"\n=== {model_name} ({variant}) ===")
    print("Accuracy :", acc)
    print("Precision:", prec)
    print("Recall   :", rec)
    print("F1       :", f1)
    if not np.isnan(roc):
        print("ROC AUC  :", roc)
    print("\nClassification report:")
    print(classification_report(y_te, y_pred))

    return {
        "Model": model_name,
        "Variant": variant,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "ROC_AUC": roc,
    }

# Sonuç listesi
results = []

# Model döngüsü
for name, clf in models.items():
    # Temel model
    baseline_pipe = make_baseline_pipeline(clf)
    res_base = evaluate_model(
        baseline_pipe, X_train, X_test, y_train, y_test, name, "No_FS"
    )
    results.append(res_base)

    # ANOVA’lı model
    anova_pipe = make_anova_pipeline(clf)
    res_anova = evaluate_model(
        anova_pipe, X_train, X_test, y_train, y_test, name, f"ANOVA_k={K}"
    )
    results.append(res_anova)


=== LogisticRegression (No_FS) ===
Accuracy : 0.6359756097560976
Precision: 0.8874560375146542
Recall   : 0.6017488076311606
F1       : 0.7171956418758882
ROC AUC  : 0.7306349312046878

Classification report:
              precision    recall  f1-score   support

           0       0.36      0.75      0.49       382
           1       0.89      0.60      0.72      1258

    accuracy                           0.64      1640
   macro avg       0.63      0.68      0.60      1640
weighted avg       0.77      0.64      0.66      1640


=== LogisticRegression (ANOVA_k=75) ===
Accuracy : 0.626219512195122
Precision: 0.8880866425992779
Recall   : 0.5866454689984102
F1       : 0.7065581617999043
ROC AUC  : 0.7262545884350629

Classification report:
              precision    recall  f1-score   support

           0       0.36      0.76      0.49       382
           1       0.89      0.59      0.71      1258

    accuracy                           0.63      1640
   macro avg       0.62      0.

9. Sonuçları Özetleme
Tüm model varyasyonlarının performans metrikleri bir DataFrame'de toplanıyor ve F1 skoruna göre büyükten küçüğe sıralanarak en iyi performans gösteren model belirleniyor.

In [9]:
# Sonuç tablosu
results_df = pd.DataFrame(results)

# Özet çıktı
print("\n\n===== Summary Results  =====")
print(results_df.sort_values(by="F1", ascending=False))



===== Summary Results  =====
                Model     Variant  Accuracy  Precision    Recall        F1  \
4           LinearSVC       No_FS  0.771341   0.775421  0.988076  0.868927   
3       SGDClassifier  ANOVA_k=75  0.770122   0.773433  0.990461  0.868595   
5           LinearSVC  ANOVA_k=75  0.769512   0.774314  0.987281  0.867925   
2       SGDClassifier       No_FS  0.768293   0.772333  0.989666  0.867596   
0  LogisticRegression       No_FS  0.635976   0.887456  0.601749  0.717196   
1  LogisticRegression  ANOVA_k=75  0.626220   0.888087  0.586645  0.706558   

    ROC_AUC  
4  0.725285  
3  0.722028  
5  0.721856  
2  0.723281  
0  0.730635  
1  0.726255  


Çıktı Yorumu: Özellik seçimi (ANOVA) olmayan LinearSVC modeli en yüksek F1 skorunu (0.8689) elde etmiş, ancak ROC AUC skoru en yüksek olan LogisticRegression (No_FS) (0.7306) olmuştur.